Question 3

Setup

In [ ]:
%pip install nltk spacy textblob svgling pandas
!{__import__('sys').executable} -m spacy download en_core_web_sm


In [10]:
import re
import nltk
import pandas as pd
from textblob import TextBlob

for pkg in ["punkt", "punkt_tab", "averaged_perceptron_tagger",
            "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)

file_path = r"C:\Users\TAI KOK WAI\Downloads\Data_2.txt"

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

sentences = nltk.sent_tokenize(text)
for i, s in enumerate(sentences):
    print(i, s)

0 The big black dog barked at the white cat and chased away.


In [11]:
sentence = sentences[0]
tokens = nltk.word_tokenize(sentence)

print("Sentence:", sentence)
print("Tokens  :", tokens)
print("Total tokens:", len(tokens))

Sentence: The big black dog barked at the white cat and chased away.
Tokens  : ['The', 'big', 'black', 'dog', 'barked', 'at', 'the', 'white', 'cat', 'and', 'chased', 'away', '.']
Total tokens: 13


========================================================================================================

3.1 POS TAGGING

3.1.1 NLTK POS TAGGER

In [12]:
nltk_tags = nltk.pos_tag(tokens)
print(nltk_tags)

# Token-by-token proof with tag meaning
tag_meaning = {
    "NN": "noun, singular", "NNS": "noun, plural", "NNP": "proper noun, singular",
    "VB": "verb, base", "VBD": "verb, past tense", "VBG": "verb, gerund/present participle",
    "VBN": "verb, past participle", "VBP": "verb, non-3rd person present", "VBZ": "verb, 3rd person present",
    "DT": "determiner", "IN": "preposition/conjunction", "JJ": "adjective", "RB": "adverb",
    "PRP": "personal pronoun", "PRP$": "possessive pronoun", "CC": "coordinating conjunction",
    "CD": "number", "TO": "to", "MD": "modal", ".": "punctuation", ",": "comma",
}
print()
for w, t in nltk_tags:
    print(f"{w:15} -> {t:5} ({tag_meaning.get(t, 'other')})")

[('The', 'DT'), ('big', 'JJ'), ('black', 'JJ'), ('dog', 'NN'), ('barked', 'VBD'), ('at', 'IN'), ('the', 'DT'), ('white', 'JJ'), ('cat', 'NN'), ('and', 'CC'), ('chased', 'VBD'), ('away', 'RB'), ('.', '.')]

The             -> DT    (determiner)
big             -> JJ    (adjective)
black           -> JJ    (adjective)
dog             -> NN    (noun, singular)
barked          -> VBD   (verb, past tense)
at              -> IN    (preposition/conjunction)
the             -> DT    (determiner)
white           -> JJ    (adjective)
cat             -> NN    (noun, singular)
and             -> CC    (coordinating conjunction)
chased          -> VBD   (verb, past tense)
away            -> RB    (adverb)
.               -> .     (punctuation)


========================================================================================================

3.1.2 TEXTBLOB POS TAGGER

In [13]:
blob = TextBlob(sentence)
textblob_tags = blob.tags
print(textblob_tags)
print("Total tagged tokens:", len(textblob_tags))

tb_words = [w for w, _ in textblob_tags]
missing = [w for w in tokens if w not in tb_words]
print("Tokens in NLTK but not in TextBlob output:", missing)

nltk_dict = dict(nltk_tags)
diff = [(w, nltk_dict[w], t) for w, t in textblob_tags if w in nltk_dict and nltk_dict[w] != t]
print("Tag differences (word, NLTK, TextBlob):", diff)

[('The', 'DT'), ('big', 'JJ'), ('black', 'JJ'), ('dog', 'NN'), ('barked', 'VBD'), ('at', 'IN'), ('the', 'DT'), ('white', 'JJ'), ('cat', 'NN'), ('and', 'CC'), ('chased', 'VBD'), ('away', 'RB')]
Total tagged tokens: 12
Tokens in NLTK but not in TextBlob output: ['.']
Tag differences (word, NLTK, TextBlob): []


^ NLTK TAG "."(FULL STOP) AS ONE TOKEN, TEXTBLOB DIDNT TAG THE FULL STOP 
SO DIFFERENCE IS NLTK 13 TOKEN AND TEXTBLOB 12

========================================================================================================

3.1.3 REGULAR EXPRESSION TAGGER

In [14]:
patterns = [
    (r'.*ing$', 'VBG'),
    (r'.*ed$', 'VBD'),
    (r'.*es$', 'VBZ'),
    (r'.*ly$', 'RB'),
    (r".*'s$", 'NN$'),
    (r'.*s$', 'NNS'),
    (r'^-?[0-9]+(\.[0-9]+)?$', 'CD'),
    (r'^(the|a|an|The|A|An)$', 'DT'),
    (r'^(in|on|at|with|of|by|for|from|to)$', 'IN'),
    (r'^(I|you|he|she|it|we|they)$', 'PRP'),
    (r'^[A-Z].*', 'NNP'),
    (r'[^\w\s]', '.'),
    (r'.*', 'NN'),            # default tag
]

regexp_tagger = nltk.RegexpTagger(patterns)
regex_tags = regexp_tagger.tag(tokens)
print(regex_tags)

# Proof 1: tokens that matched a specific rule vs. fell back to the default NN
nltk_dict = dict(nltk_tags)
default_used = [w for w, t in regex_tags if t == "NN" and nltk_dict.get(w) != "NN"]
print("\nTokens wrongly given the default tag NN:", [(w, nltk_dict[w]) for w in default_used])

# Proof 2: tokens the regex tagger got right (same as NLTK)
correct = [w for (w, t1), (_, t2) in zip(regex_tags, nltk_tags) if t1 == t2]
print("Tokens matching NLTK:", correct)
print(f"Agreement with NLTK: {len(correct)}/{len(tokens)} = {len(correct)/len(tokens):.0%}")

[('The', 'DT'), ('big', 'NN'), ('black', 'NN'), ('dog', 'NN'), ('barked', 'VBD'), ('at', 'IN'), ('the', 'DT'), ('white', 'NN'), ('cat', 'NN'), ('and', 'NN'), ('chased', 'VBD'), ('away', 'NN'), ('.', '.')]

Tokens wrongly given the default tag NN: [('big', 'JJ'), ('black', 'JJ'), ('white', 'JJ'), ('and', 'CC'), ('away', 'RB')]
Tokens matching NLTK: ['The', 'dog', 'barked', 'at', 'the', 'cat', 'chased', '.']
Agreement with NLTK: 8/13 = 62%


^ REGULAR EXPRESSION TAG 5 WORD DIFFERENT

========================================================================================================

3.2 EXPLNATION FOR DIFFERENCE OF POS TAGGER 

In [16]:

tb_aligned, j = [], 0
for w in tokens:
    if j < len(textblob_tags) and textblob_tags[j][0] == w:
        tb_aligned.append(textblob_tags[j][1]); j += 1
    else:
        tb_aligned.append("(not tokenised)")

comparison = pd.DataFrame({
    "Token": tokens,
    "NLTK": [t for _, t in nltk_tags],
    "TextBlob": tb_aligned,
    "Regex": [t for _, t in regex_tags],
})

def result(row):
    if row["TextBlob"] == "(not tokenised)":
        tb = "TextBlob missing token"
    else:
        tb = "NLTK=TextBlob" if row["NLTK"] == row["TextBlob"] else "NLTK≠TextBlob"
    rx = "NLTK=Regex" if row["NLTK"] == row["Regex"] else "NLTK≠Regex"
    return pd.Series([tb, rx])

comparison[["NLTK vs TextBlob", "NLTK vs Regex"]] = comparison.apply(result, axis=1)
comparison

,Token,NLTK,TextBlob,Regex,NLTK vs TextBlob,NLTK vs Regex
0,The,DT,DT,DT,NLTK=TextBlob,NLTK=Regex
1,big,JJ,JJ,NN,NLTK=TextBlob,NLTK≠Regex
2,black,JJ,JJ,NN,NLTK=TextBlob,NLTK≠Regex
3,dog,NN,NN,NN,NLTK=TextBlob,NLTK=Regex
4,barked,VBD,VBD,VBD,NLTK=TextBlob,NLTK=Regex
5,at,IN,IN,IN,NLTK=TextBlob,NLTK=Regex
6,the,DT,DT,DT,NLTK=TextBlob,NLTK=Regex
7,white,JJ,JJ,NN,NLTK=TextBlob,NLTK≠Regex
8,cat,NN,NN,NN,NLTK=TextBlob,NLTK=Regex
9,and,CC,CC,NN,NLTK=TextBlob,NLTK≠Regex


EXPLANATION:

NLTK AND TEXTBLOB TOKEN THE SAME AMOUNT OF WORD, JUST HAVE DIFFERENCE IN TOKENIZE THE FULL STOP SIGN

REGULAR EXPRESSION HAVE DIFFERENCE WITH NLTK(5 WORD OUT OF 13), BECAUSE IT USE FIX RULE WITHOUT CONTEXT AND DEFAULT TO NN

========================================================================================================

3.3 PARSE TREE FOR DATA_2

In [18]:

import re
import nltk
import svgling
from collections import defaultdict
from IPython.display import display


words = [w for w in tokens if re.match(r"\w", w)]
pos_words = nltk.pos_tag(words)

print("Tokens from NLTK tokenizer :", tokens)
print("Words used for parsing     :", words)
print("Removed before parsing     :", [w for w in tokens if w not in words])
print("NLTK POS tags of the words :", pos_words)


def category(tag):
    if tag.startswith("NN"): return "N"
    if tag.startswith("VB"): return "V"
    if tag in ("DT", "PRP$"): return "Det"
    if tag.startswith("JJ"): return "Adj"
    if tag in ("IN", "TO"): return "P"
    if tag == "PRP": return "Pro"
    if tag.startswith("RB"): return "Adv"
    if tag == "CC": return "Conj"
    return None

lexicon = defaultdict(set)
for w, t in pos_words:
    c = category(t)
    if c:
        lexicon[c].add(w)

structure = """
S  -> NP VP | S Conj S
NP -> Det N | Det Adj N | Pro | N | Adj N | NP PP | NP Conj NP
VP -> V | V NP | V PP | V NP PP | VP PP | V Adv
PP -> P NP
"""
lexical = "\n".join(f'{c} -> ' + " | ".join(f'"{w}"' for w in sorted(ws))
                    for c, ws in lexicon.items())

grammar_str = structure + lexical
print("\n===== Grammar =====")
print(grammar_str)


grammar = nltk.CFG.fromstring(grammar_str)
parser = nltk.ChartParser(grammar)
trees = list(parser.parse(words))
print("\nNumber of possible parse trees:", len(trees))


for i, t in enumerate(trees, 1):
    print(f"\n===== Parse tree {i} =====")
    print(t)                 # bracket form
    t.pretty_print()         # text drawing
    display(svgling.draw_tree(t))   # graphical drawing

    print("Phrases in this tree:")
    for sub in t.subtrees(lambda x: x.label() in ("NP", "VP", "PP")):
        print(f"  {sub.label():3} -> {' '.join(sub.leaves())}")

Tokens from NLTK tokenizer : ['The', 'big', 'black', 'dog', 'barked', 'at', 'the', 'white', 'cat', 'and', 'chased', 'away', '.']
Words used for parsing     : ['The', 'big', 'black', 'dog', 'barked', 'at', 'the', 'white', 'cat', 'and', 'chased', 'away']
Removed before parsing     : ['.']
NLTK POS tags of the words : [('The', 'DT'), ('big', 'JJ'), ('black', 'JJ'), ('dog', 'NN'), ('barked', 'VBD'), ('at', 'IN'), ('the', 'DT'), ('white', 'JJ'), ('cat', 'NN'), ('and', 'CC'), ('chased', 'VBD'), ('away', 'RB')]

===== Grammar =====

S  -> NP VP | S Conj S
NP -> Det N | Det Adj N | Pro | N | Adj N | NP PP | NP Conj NP
VP -> V | V NP | V PP | V NP PP | VP PP | V Adv
PP -> P NP
Det -> "The" | "the"
Adj -> "big" | "black" | "white"
N -> "cat" | "dog"
V -> "barked" | "chased"
P -> "at"
Conj -> "and"
Adv -> "away"

Number of possible parse trees: 0


========================================================================================================

========================================================================================================

QUESTION 5 (MOSES TOKENIZER)

SETUP

In [ ]:
%pip install sacremoses 
#(REWRITE IN PYTHON BUT STILL CONSIDER AS MOSES)
# %pip install mosestokenizer (CANNOT USE CUZ MOSES IS PERL VERSION, NEED MORE STEP IF WANT TO USE)

REFERENCE:

https://pypi.org/project/mosestokenizer/

https://pypi.org/project/sacremoses/

========================================================================================================

CODE

In [4]:
from sacremoses import MosesTokenizer

with open(r"C:\Users\TAI KOK WAI\Downloads\Data_1.txt", "r", encoding="utf-8") as file:
    text = file.read()

moses_tokenizer = MosesTokenizer(lang="en")
moses_tokens = moses_tokenizer.tokenize(text, escape=False)

print("Moses tokens:")
print(moses_tokens)
print("\nMoses token count:", len(moses_tokens))
print("NLTK token count: 94")

Moses tokens:
['Classification', 'is', 'the', 'task', 'of', 'choosing', 'the', 'correct', 'class', 'label', 'for', 'a', 'given', 'input', '.', 'In', 'basic', 'classification', 'tasks', ',', 'each', 'input', 'is', 'considered', 'in', 'isolation', 'from', 'all', 'other', 'inputs', ',', 'and', 'the', 'set', 'of', 'labels', 'is', 'defined', 'in', 'advance', '.', 'The', 'basic', 'classification', 'task', 'has', 'a', 'number', 'of', 'interesting', 'variants', '.', 'For', 'example', ',', 'in', 'multiclass', 'classification', ',', 'each', 'instance', 'may', 'be', 'assigned', 'multiple', 'labels', ';', 'in', 'open-class', 'classification', ',', 'the', 'set', 'of', 'labels', 'is', 'not', 'defined', 'in', 'advance', ';', 'and', 'in', 'sequence', 'classification', ',', 'a', 'list', 'of', 'inputs', 'are', 'jointly', 'classified', '.']

Moses token count: 94
NLTK token count: 94


DATA IS TOO SIMPLE, SO BOTH IS SAME

SO I USE MY OWN PARAGRAPH TTO DIFFERENTIATE BOTH

========================================================================================================

In [6]:
from sacremoses import MosesTokenizer
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

paragraph = (
    "Don't split e-mail addresses like john.doe@example.com. "
    "Visit https://example.com/page?id=5 for details. "
    "The U.S.A. spent $3.14 million, or 25% of the \"well-known\" budget; "
    "it's Dr. Smith's decision (not ours)."
)

moses_tokenizer = MosesTokenizer(lang="en")
moses_tokens = moses_tokenizer.tokenize(paragraph, escape=False)
nltk_tokens = nltk.word_tokenize(paragraph)

print("Moses tokens:")
print(moses_tokens)
print("\nMoses token count:", len(moses_tokens))
print("\nNLTK tokens:")
print(nltk_tokens)
print("\nNLTK token count:", len(nltk_tokens))

Moses tokens:
['Don', "'t", 'split', 'e-mail', 'addresses', 'like', 'john.doe', '@', 'example.com.', 'Visit', 'https', ':', '/', '/', 'example.com', '/', 'page', '?', 'id', '=', '5', 'for', 'details', '.', 'The', 'U.S.A.', 'spent', '$', '3.14', 'million', ',', 'or', '25', '%', 'of', 'the', '"', 'well-known', '"', 'budget', ';', 'it', "'s", 'Dr.', 'Smith', "'s", 'decision', '(', 'not', 'ours', ')', '.']

Moses token count: 52

NLTK tokens:
['Do', "n't", 'split', 'e-mail', 'addresses', 'like', 'john.doe', '@', 'example.com', '.', 'Visit', 'https', ':', '//example.com/page', '?', 'id=5', 'for', 'details', '.', 'The', 'U.S.A.', 'spent', '$', '3.14', 'million', ',', 'or', '25', '%', 'of', 'the', '``', 'well-known', "''", 'budget', ';', 'it', "'s", 'Dr.', 'Smith', "'s", 'decision', '(', 'not', 'ours', ')', '.']

NLTK token count: 47
